# 01 — End-to-End Pipeline Test

Tests the full PyTorch pipeline:  
**Data → Pairs → Model → Training → Evaluation**

Dataset used here: **CEDAR** (`data/raw/CEDAR/`)  
Once each section passes, the logic will be promoted into `src/`.

---
## Section 0 — Setup

In [ ]:
import sys
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

from sklearn import metrics as sk_metrics

# Make src/ importable from the notebook
PROJECT_ROOT = Path("../").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.seed import set_seed
from src.data.transforms import get_default_transforms
from src.models.backbones import SmallCNN, ResNet18Embed
from src.losses.contrastive import ContrastiveLoss
from src.losses.triplet import TripletLoss
from src.metrics.verification import compute_metrics

SEED = 42
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("Project root:", PROJECT_ROOT)

---
## Section 1 — Dataset EDA (CEDAR)

In [ ]:
# ── Edit this path to match where you placed CEDAR ──────────────────────────
DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "CEDAR"
ORG_DIR   = DATA_ROOT / "full_org"
FORG_DIR  = DATA_ROOT / "full_forg"

print("ORG_DIR  exists:", ORG_DIR.exists(),  ORG_DIR)
print("FORG_DIR exists:", FORG_DIR.exists(), FORG_DIR)

In [ ]:
PAT_ORG  = re.compile(r"^original_(\d+)_(\d+)\.png$",  re.IGNORECASE)
PAT_FORG = re.compile(r"^forgeries_(\d+)_(\d+)\.png$", re.IGNORECASE)

def scan_cedar(org_dir: Path, forg_dir: Path) -> pd.DataFrame:
    rows = []
    for fp in org_dir.iterdir():
        m = PAT_ORG.match(fp.name)
        if m:
            rows.append({"path": str(fp), "writer_id": int(m.group(1)),
                         "sample_id": int(m.group(2)), "label": "genuine"})
    for fp in forg_dir.iterdir():
        m = PAT_FORG.match(fp.name)
        if m:
            rows.append({"path": str(fp), "writer_id": int(m.group(1)),
                         "sample_id": int(m.group(2)), "label": "forgery"})
    return pd.DataFrame(rows).sort_values(["writer_id", "label", "sample_id"]).reset_index(drop=True)

df = scan_cedar(ORG_DIR, FORG_DIR)

print(f"Total images : {len(df)}")
print(df["label"].value_counts().to_string())
print(f"Unique writers: {df['writer_id'].nunique()}")
df.head()

In [ ]:
# Sanity check: CEDAR has 55 writers × 24 genuine + 55 × 24 forgery = 2640
assert len(df) == 2640, f"Expected 2640, got {len(df)}"
assert df["writer_id"].nunique() == 55
print("Assertions passed.")

In [ ]:
# Visualise sample signatures: 3 random writers, 2 genuine + 2 forgery each
rng = np.random.default_rng(SEED)
sample_writers = rng.choice(df["writer_id"].unique(), size=3, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(14, 9))
fig.suptitle("Sample Signatures — 3 Writers (genuine | genuine | forgery | forgery)", fontsize=12)

for row_idx, wid in enumerate(sample_writers):
    sub = df[df["writer_id"] == wid]
    gen  = sub[sub["label"] == "genuine"]["path"].tolist()
    forg = sub[sub["label"] == "forgery"]["path"].tolist()
    for col_idx, path in enumerate(gen[:2] + forg[:2]):
        ax = axes[row_idx, col_idx]
        ax.imshow(Image.open(path), cmap="gray")
        kind = "GEN" if col_idx < 2 else "FORG"
        ax.set_title(f"W{wid} {kind}", fontsize=8)
        ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Per-writer count bar chart
counts = df.groupby(["writer_id", "label"]).size().unstack(fill_value=0)
counts.plot(kind="bar", figsize=(16, 4), width=0.8, color=["steelblue", "salmon"])
plt.title("Samples per writer")
plt.xlabel("Writer ID")
plt.ylabel("Count")
plt.xticks(rotation=90, fontsize=6)
plt.legend(["Forgery", "Genuine"])
plt.tight_layout()
plt.show()

---
## Section 2 — Writer-level Train / Val / Test Split

Split by **writer** so no writer appears in more than one partition.  
Ratios: 70 / 15 / 15

In [ ]:
def split_writers(df: pd.DataFrame, train=0.70, val=0.15, test=0.15, seed=42):
    assert abs(train + val + test - 1.0) < 1e-9
    writers = np.array(sorted(df["writer_id"].unique()))
    rng = np.random.default_rng(seed)
    rng.shuffle(writers)
    n = len(writers)
    n_train = int(round(n * train))
    n_val   = int(round(n * val))
    return set(writers[:n_train]), set(writers[n_train:n_train+n_val]), set(writers[n_train+n_val:])

train_writers, val_writers, test_writers = split_writers(df, seed=SEED)

print(f"Train writers : {len(train_writers)}")
print(f"Val   writers : {len(val_writers)}")
print(f"Test  writers : {len(test_writers)}")

# Verify no leakage
assert not (train_writers & val_writers),  "Train/Val overlap!"
assert not (train_writers & test_writers), "Train/Test overlap!"
assert not (val_writers   & test_writers), "Val/Test overlap!"
print("No overlap — OK.")

---
## Section 3 — Pair Generation

Label convention (matches contrastive loss in `src/losses/contrastive.py`):  
- `label = 1` → positive (genuine-genuine, **same writer**)  
- `label = 0` → negative (genuine-forgery same writer **or** genuine-genuine cross-writer)

In [ ]:
def build_pools(df: pd.DataFrame, writer_set: set):
    sub = df[df["writer_id"].isin(writer_set)]
    genuine_by_writer = {}
    forgery_by_writer = {}
    for wid, grp in sub.groupby("writer_id"):
        g = grp[grp["label"] == "genuine"]["path"].tolist()
        f = grp[grp["label"] == "forgery"]["path"].tolist()
        if g: genuine_by_writer[wid] = g
        if f: forgery_by_writer[wid] = f
    return genuine_by_writer, forgery_by_writer


def generate_pairs(df: pd.DataFrame, writer_set: set,
                   n_pairs: int = 20_000, seed: int = 1,
                   neg_mix: float = 0.5) -> pd.DataFrame:
    """
    neg_mix: fraction of negatives that are genuine-forgery (same writer).
             the rest are genuine-genuine cross-writer.
    """
    genuine_by_writer, forgery_by_writer = build_pools(df, writer_set)
    writers = sorted(genuine_by_writer.keys())
    writers_with_forg = sorted(set(genuine_by_writer) & set(forgery_by_writer))

    if len(writers) < 2:
        raise ValueError("Need >=2 writers")
    if not writers_with_forg:
        raise ValueError("Need writers with both genuine and forgery samples")

    rng = np.random.default_rng(seed)
    n_pos = n_pairs // 2
    n_neg = n_pairs - n_pos
    n_neg_same  = int(round(n_neg * neg_mix))
    n_neg_cross = n_neg - n_neg_same

    rows = []

    # Positives: genuine-genuine same writer
    for _ in range(n_pos):
        w = rng.choice(writers)
        g = genuine_by_writer[w]
        i, j = rng.choice(len(g), size=2, replace=False)
        rows.append({"path_a": g[i], "path_b": g[j], "label": 1,
                     "pair_type": "pos", "writer_a": w, "writer_b": w})

    # Negatives: genuine-forgery same writer
    for _ in range(n_neg_same):
        w = rng.choice(writers_with_forg)
        g = genuine_by_writer[w]
        f = forgery_by_writer[w]
        rows.append({"path_a": g[rng.integers(len(g))],
                     "path_b": f[rng.integers(len(f))],
                     "label": 0, "pair_type": "neg_same_writer",
                     "writer_a": w, "writer_b": w})

    # Negatives: genuine-genuine cross-writer
    for _ in range(n_neg_cross):
        w1, w2 = rng.choice(writers, size=2, replace=False)
        g1, g2 = genuine_by_writer[w1], genuine_by_writer[w2]
        rows.append({"path_a": g1[rng.integers(len(g1))],
                     "path_b": g2[rng.integers(len(g2))],
                     "label": 0, "pair_type": "neg_cross_writer",
                     "writer_a": w1, "writer_b": w2})

    return pd.DataFrame(rows).sample(frac=1.0, random_state=seed).reset_index(drop=True)


train_pairs = generate_pairs(df, train_writers, n_pairs=40_000, seed=1, neg_mix=0.8)
val_pairs   = generate_pairs(df, val_writers,   n_pairs=10_000, seed=2, neg_mix=0.8)
test_pairs  = generate_pairs(df, test_writers,  n_pairs=10_000, seed=3, neg_mix=0.8)

print(f"Train: {len(train_pairs):,}   Val: {len(val_pairs):,}   Test: {len(test_pairs):,}")
print("Train label balance:")
print(train_pairs["label"].value_counts(normalize=True).to_string())
train_pairs.head()

In [ ]:
# Sanity checks on pair consistency
pos_bad   = train_pairs[(train_pairs["label"] == 1) & (train_pairs["writer_a"] != train_pairs["writer_b"])]
cross_bad = train_pairs[(train_pairs["pair_type"] == "neg_cross_writer") & (train_pairs["writer_a"] == train_pairs["writer_b"])]
print("Positives with different writers (should be 0):", len(pos_bad))
print("Cross-writer negatives with same writer (should be 0):", len(cross_bad))
assert len(pos_bad) == 0 and len(cross_bad) == 0
print("All pair sanity checks passed.")

---
## Section 4 — PyTorch Dataset & DataLoader

> **Bug noted**: `src/data/pairs.py::PairDataset.__getitem__` returns `a[0]` which is a path string — it never loads the image.  
> `SiamesePairDataset` below fixes this inline.

In [ ]:
class SiamesePairDataset(Dataset):
    """Loads (img_a, img_b, label) pairs from a DataFrame of file paths."""

    def __init__(self, pairs_df: pd.DataFrame, transform=None):
        self.pairs     = pairs_df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        row   = self.pairs.iloc[idx]
        img_a = Image.open(row["path_a"]).convert("L")
        img_b = Image.open(row["path_b"]).convert("L")
        if self.transform:
            img_a = self.transform(img_a)
            img_b = self.transform(img_b)
        label = torch.tensor(row["label"], dtype=torch.float32)
        return img_a, img_b, label


IMG_SIZE  = 224
BATCH     = 32

tfm = get_default_transforms(IMG_SIZE)

train_ds = SiamesePairDataset(train_pairs, transform=tfm)
val_ds   = SiamesePairDataset(val_pairs,   transform=tfm)
test_ds  = SiamesePairDataset(test_pairs,  transform=tfm)

# num_workers=0 for Windows compatibility
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=DEVICE.type=="cuda")
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=DEVICE.type=="cuda")
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=DEVICE.type=="cuda")

# Verify shapes
a, b, y = next(iter(train_loader))
print("img_a shape:", a.shape)
print("img_b shape:", b.shape)
print("label shape:", y.shape, "  unique:", y.unique())
assert a.shape == torch.Size([BATCH, 1, IMG_SIZE, IMG_SIZE])
print("DataLoader shape check passed.")

---
## Section 5 — Model Sanity Check

In [ ]:
EMB_DIM_SMALL   = 128
EMB_DIM_RESNET  = 256

small_cnn  = SmallCNN(emb_dim=EMB_DIM_SMALL).to(DEVICE)
resnet18   = ResNet18Embed(emb_dim=EMB_DIM_RESNET, pretrained=False).to(DEVICE)

x_test = a.to(DEVICE)

with torch.no_grad():
    out_small  = small_cnn(x_test)
    out_resnet = resnet18(x_test)

print("SmallCNN   output:", out_small.shape,  " (expected [32, 128])")
print("ResNet18   output:", out_resnet.shape, " (expected [32, 256])")

assert out_small.shape  == torch.Size([BATCH, EMB_DIM_SMALL])
assert out_resnet.shape == torch.Size([BATCH, EMB_DIM_RESNET])

n_small  = sum(p.numel() for p in small_cnn.parameters() if p.requires_grad)
n_resnet = sum(p.numel() for p in resnet18.parameters()  if p.requires_grad)
print(f"\nSmallCNN   params : {n_small:,}")
print(f"ResNet18   params : {n_resnet:,}")
print("Model shape checks passed.")

---
## Section 6 — Loss Functions

In [ ]:
criterion_contrastive = ContrastiveLoss(margin=1.0)

# Dummy forward pass
with torch.no_grad():
    e1 = small_cnn(a.to(DEVICE))
    e2 = small_cnn(b.to(DEVICE))
    lbl = y.to(DEVICE)
    loss_val = criterion_contrastive(e1, e2, lbl)

print("ContrastiveLoss value:", loss_val.item())
assert loss_val.ndim == 0, "Loss must be a scalar"
print("ContrastiveLoss check passed.")

In [ ]:
criterion_triplet = TripletLoss(margin=1.0)

# Dummy triplet forward pass (anchor, positive, negative)
with torch.no_grad():
    anchor   = small_cnn(a.to(DEVICE))
    positive = small_cnn(b.to(DEVICE))
    negative = small_cnn(a.flip(0).to(DEVICE))  # just a cheap stand-in
    loss_trip = criterion_triplet(anchor, positive, negative)

print("TripletLoss value:", loss_trip.item())
assert loss_trip.ndim == 0
print("TripletLoss check passed.")

---
## Section 7 — Mini Training Loop (3 Epochs)

Uses `SmallCNN` + `ContrastiveLoss` + `Adam`.  
Tracks `loss`, `mean_pos_dist`, `mean_neg_dist` per epoch.

In [ ]:
set_seed(SEED)

encoder   = SmallCNN(emb_dim=EMB_DIM_SMALL).to(DEVICE)
criterion = ContrastiveLoss(margin=1.0)
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-4)

N_EPOCHS = 3
history  = {"loss": [], "mean_pos_dist": [], "mean_neg_dist": []}

for epoch in range(N_EPOCHS):
    encoder.train()
    epoch_loss  = 0.0
    pos_dists   = []
    neg_dists   = []
    n_batches   = 0

    for img_a, img_b, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{N_EPOCHS}", leave=False):
        img_a, img_b, labels = img_a.to(DEVICE), img_b.to(DEVICE), labels.to(DEVICE)

        e1 = encoder(img_a)
        e2 = encoder(img_b)

        loss = criterion(e1, e2, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            dist = F.pairwise_distance(e1, e2)  # Euclidean
            pos_dists.append(dist[labels == 1].cpu())
            neg_dists.append(dist[labels == 0].cpu())

        epoch_loss += loss.item()
        n_batches  += 1

    avg_loss = epoch_loss / n_batches
    all_pos  = torch.cat(pos_dists).mean().item() if pos_dists else float("nan")
    all_neg  = torch.cat(neg_dists).mean().item() if neg_dists else float("nan")

    history["loss"].append(avg_loss)
    history["mean_pos_dist"].append(all_pos)
    history["mean_neg_dist"].append(all_neg)

    print(f"Epoch {epoch+1:02d}/{N_EPOCHS}  loss={avg_loss:.4f}  "
          f"mean_pos_dist={all_pos:.4f}  mean_neg_dist={all_neg:.4f}")

In [ ]:
epochs_x = range(1, N_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_x, history["loss"], marker="o", color="royalblue")
ax1.set_title("Training Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("ContrastiveLoss")
ax1.grid(True)

ax2.plot(epochs_x, history["mean_pos_dist"], marker="o", label="Pos (same writer)",  color="green")
ax2.plot(epochs_x, history["mean_neg_dist"], marker="s", label="Neg (forgery/cross)", color="red")
ax2.set_title("Mean Euclidean Distance per Class")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Distance")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

print("After training: neg_dist > pos_dist?", history["mean_neg_dist"][-1] > history["mean_pos_dist"][-1])

---
## Section 8 — Evaluation

Compute pairwise Euclidean distances on the test set,  
then derive AUC, EER, FAR, FRR via `src.metrics.verification.compute_metrics`.

In [ ]:
encoder.eval()
all_dists  = []
all_labels = []

with torch.no_grad():
    for img_a, img_b, labels in tqdm(test_loader, desc="Evaluating"):
        img_a, img_b = img_a.to(DEVICE), img_b.to(DEVICE)
        e1 = encoder(img_a)
        e2 = encoder(img_b)
        dist = F.pairwise_distance(e1, e2).cpu().numpy()
        all_dists.append(dist)
        all_labels.append(labels.numpy())

distances = np.concatenate(all_dists)
y_true    = np.concatenate(all_labels).astype(int)

print("Distances  shape:", distances.shape)
print("Labels     shape:", y_true.shape)
print("Mean dist (pos):", distances[y_true == 1].mean().round(4))
print("Mean dist (neg):", distances[y_true == 0].mean().round(4))

In [ ]:
# compute_metrics expects *similarity* (higher = more similar)
# convert Euclidean distance to similarity score
similarity = 1.0 / (1.0 + distances)

results = compute_metrics(y_true, similarity)

print("\n=== Test Metrics ===")
for k, v in results.items():
    print(f"  {k:<20s}: {v:.4f}")

In [ ]:
# ROC curve
fpr, tpr, _ = sk_metrics.roc_curve(y_true, similarity)
auc_val = results["auc"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Plot 1: ROC
axes[0].plot(fpr, tpr, color="royalblue", lw=2, label=f"AUC = {auc_val:.3f}")
axes[0].plot([0, 1], [0, 1], "k--")
axes[0].set_xlabel("FPR")
axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curve")
axes[0].legend()
axes[0].grid(True)

# Plot 2: FAR / FRR vs threshold
thresholds  = np.linspace(similarity.min(), similarity.max(), 400)
far_arr, frr_arr = [], []
for t in thresholds:
    preds = (similarity >= t).astype(int)
    tp = int(((preds == 1) & (y_true == 1)).sum())
    fp = int(((preds == 1) & (y_true == 0)).sum())
    fn = int(((preds == 0) & (y_true == 1)).sum())
    tn = int(((preds == 0) & (y_true == 0)).sum())
    far_arr.append(fp / (fp + tn) if (fp + tn) > 0 else 0.0)
    frr_arr.append(fn / (fn + tp) if (fn + tp) > 0 else 0.0)

far_arr = np.array(far_arr)
frr_arr = np.array(frr_arr)
eer_thresh = results["eer_threshold"]

axes[1].plot(thresholds, far_arr, label="FAR", color="red")
axes[1].plot(thresholds, frr_arr, label="FRR", color="green")
axes[1].axvline(eer_thresh, linestyle="--", color="gray", label=f"EER thr={eer_thresh:.3f}")
axes[1].set_xlabel("Similarity threshold")
axes[1].set_ylabel("Rate")
axes[1].set_title(f"FAR / FRR  (EER={results['eer']:.3f})")
axes[1].legend()
axes[1].grid(True)

# Plot 3: Distance distributions
axes[2].hist(distances[y_true == 1], bins=50, alpha=0.6, color="green",  label="Positive (same writer)")
axes[2].hist(distances[y_true == 0], bins=50, alpha=0.6, color="red",    label="Negative (forgery/cross)")
axes[2].set_xlabel("Euclidean distance")
axes[2].set_ylabel("Count")
axes[2].set_title("Distance Distribution")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

---
## Section 9 — Summary

In [ ]:
from IPython.display import Markdown, display

rows_md = "\n".join(
    f"| {k:<22s} | {v:.4f} |"
    for k, v in results.items()
)

md = f"""
### Test Metrics — CEDAR baseline (SmallCNN, 3 epochs)

| Metric                 | Value  |
|:-----------------------|-------:|
{rows_md}

### Bugs found in `src/` template

| File | Bug | Fix needed |
|------|-----|------------|
| `src/data/pairs.py:PairDataset.__getitem__` | Returns `a[0]` (path string) — never loads image | Add `Image.open(...).convert("L")` + apply transform |
| `train.py:52-65` | Pairs consecutive items in a raw batch (not proper siamese pairs) | Replace with `SiamesePairDataset` + `DataLoader` |
| `src/data/datasets.py:_scan` | Only handles CEDAR flat-file naming; SigComp/MCYT-100 differ | Add per-dataset scanner dispatch |

### `src/` functions confirmed working
- `src.utils.seed.set_seed` ✓  
- `src.data.transforms.get_default_transforms` ✓  
- `src.models.backbones.SmallCNN` / `ResNet18Embed` ✓  
- `src.losses.contrastive.ContrastiveLoss` ✓  
- `src.losses.triplet.TripletLoss` ✓  
- `src.metrics.verification.compute_metrics` ✓  
"""

display(Markdown(md))